# NAIP COG Builder (GDAL-First)

This notebook converts unzipped NAIP imagery into **Cloud Optimized GeoTIFFs (COGs)** for two visualization modes and assembles aggregate VRTs across all images.

- **RGB** — bands `1,2,3` (natural color)
- **IRG** — bands `4,1,2` (infrared / CIR false color)

The implementation uses `osgeo.gdal` and `osgeo.osr` throughout.

In [1]:
# ── Environment preflight check ──────────────────────────────────────────────
# Run this cell FIRST to catch GDAL / Python-binding mismatches before they
# produce cryptic dlopen errors later in the workflow.
import subprocess, sys, importlib.util

def _check_gdal_env():
    issues = []
    gdal_ver = None

    # 1. Verify osgeo is importable at all.
    spec = importlib.util.find_spec('osgeo')
    if spec is None:
        issues.append('osgeo is not installed in this Python environment.')
    else:
        # Catch both ImportError (ModuleNotFoundError) and OSError (dlopen failure).
        try:
            from osgeo import gdal
            gdal_ver = gdal.__version__
        except (ImportError, OSError) as e:
            # ImportError / ModuleNotFoundError: osgeo package present but .so missing
            #   or a dependent .so (e.g. libgdal.38.dylib) cannot be found.
            # OSError: library loaded but something else went wrong at the C level.
            #
            # Most common cause: pip-installed GDAL bindings compiled against a
            # libgdal version absent from this env (e.g. libgdal.38 vs .37).
            #   Fix:  pip uninstall gdal -y
            #         conda install -n <env> -c conda-forge gdal=<version> -y
            #         (then restart the kernel)
            issues.append(
                f'osgeo import failed ({type(e).__name__}): {e}\n'
                '  Likely cause: pip-installed GDAL bindings compiled against a\n'
                '  libgdal version that does not exist in this environment.\n'
                '  Fix:\n'
                '    pip uninstall gdal -y\n'
                '    conda install -n <env> -c conda-forge gdal=<version> -y\n'
                '  Then restart the kernel.'
            )

    if gdal_ver:
        # 2. Require GDAL >= 3.4 for the OVERVIEWS=AUTO COG creation option.
        major, minor, *_ = (int(x) for x in gdal_ver.split('.'))
        if (major, minor) < (3, 4):
            issues.append(
                f'GDAL {gdal_ver} detected. OVERVIEWS=AUTO requires GDAL >= 3.4.'
            )
        else:
            print(f'GDAL version : {gdal_ver}  OK')

    # 3. Warn if a pip GDAL package is present alongside a conda one.
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', 'gdal'],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode == 0:
            pip_ver = next(
                (l.split(':', 1)[1].strip()
                 for l in result.stdout.splitlines()
                 if l.lower().startswith('version')),
                'unknown',
            )
            issues.append(
                f'pip-installed GDAL {pip_ver} is present in this environment.\n'
                '  This shadows conda-provided bindings and causes dlopen failures.\n'
                '  Fix:  pip uninstall gdal -y  (then restart the kernel)'
            )
    except Exception:
        pass

    if issues:
        print('=== GDAL environment issues ===')
        for i, msg in enumerate(issues, 1):
            print(f'[{i}] {msg}')
    else:
        print('Environment check passed.')

_check_gdal_env()

GDAL version : 3.11.4  OK
=== GDAL environment issues ===
[1] pip-installed GDAL 3.11.4 is present in this environment.
  This shadows conda-provided bindings and causes dlopen failures.
  Fix:  pip uninstall gdal -y  (then restart the kernel)


## What This Notebook Does

1. Scans one or more NAIP root directories for TIFF imagery.
2. For each image, calculates:
   - Ground sample distance (GSD) in meters/pixel.
   - Native full-resolution web zoom level.
   - Powers-of-2 overview levels for efficient multiscale reads.
3. Creates one **canonical COG per source TIFF** using the full set of source bands (no band dropping by default).
4. Optionally builds aggregate visualization VRTs (`rgb_all.vrt`, `irg_all.vrt`) from those canonical COGs.
5. Consolidates written COGs into root-level phase folders:
   - `<NAIP_ROOT>/cogs/prefire`
   - `<NAIP_ROOT>/cogs/postfire`

This keeps source information intact while still supporting visualization band remapping at the VRT layer.

## Cell 4: Configuration

Edit the values in the code cell below before running anything else.

| Variable | Purpose |
|---|---|
| `NAIP_ROOTS` | List of paths to scan; add/uncomment entries for multiple fires. |
| `FALLBACK_EPSG` | EPSG code used when imagery lacks an embedded CRS. |
| `COG_BLOCK_SIZE` | Internal tile size for COGs (512 is the de-facto standard). |
| `OVERVIEW_RESAMPLING` | Resampling method used when building overviews (`AVERAGE` for imagery). |
| `COG_COMPRESS` | Compression codec (`DEFLATE` for lossless; `WEBP`/`LERC` for lossy). |
| `COG_PREDICTOR` | TIFF predictor value (usually `2` for integer imagery). |
| `DRY_RUN` | `True` = report only; `False` = write COG and VRT files. |
| `OVERWRITE_OUTPUT` | Whether to overwrite existing COG output files. |
| `BUILD_VIZ_VRTS` | Build aggregate RGB/IRG VRTs from canonical 4-band COGs. |

In [2]:
from pathlib import Path
import math
from dataclasses import dataclass
from typing import Iterable

from osgeo import gdal, osr

gdal.UseExceptions()

# ----------------------------- User Configuration ----------------------------
# One or more root folders containing unzipped NAIP imagery.
NAIP_ROOTS = [
    Path('./downloads/castle/naip'),
    # Path('./downloads/creek/naip'),
    # Path('./downloads/czu/naip'),
    # Path('./downloads/northcomplex/naip'),
]

# Raster extensions to include when scanning.
IMAGE_EXTENSIONS = {'.tif', '.tiff'}

# Fallback EPSG if imagery lacks an embedded CRS (set to None to disable).
FALLBACK_EPSG = 26911

# Internal tile block size for COG output (512 px is the de-facto standard).
COG_BLOCK_SIZE = 512

# Resampling used when building overview levels.
# AVERAGE smooths well for continuous imagery; NEAREST preserves exact values.
OVERVIEW_RESAMPLING = 'AVERAGE'

# Compression and horizontal-differencing predictor for COG output.
# DEFLATE + PREDICTOR=2 gives efficient lossless compression for integer imagery.
COG_COMPRESS = 'DEFLATE'
COG_PREDICTOR = '2'

# Sub-folder under each source image's parent directory for initial COG output.
# A later cell consolidates these to <root>/cogs/prefire or <root>/cogs/postfire.
COGS_DIRNAME = 'cogs'

# Optional viz-band aggregate VRT outputs built from canonical COGs.
BUILD_VIZ_VRTS = True
RGB_BANDS = (1, 2, 3)  # Natural color
IRG_BANDS = (4, 1, 2)  # Infrared, Red, Green (CIR false color)
AGGREGATE_DIR = NAIP_ROOTS[0] / 'cogs_aggregates'
RGB_AGGREGATE_VRT = AGGREGATE_DIR / 'rgb_all.vrt'
IRG_AGGREGATE_VRT = AGGREGATE_DIR / 'irg_all.vrt'

# Set DRY_RUN = False to actually write COG and VRT files.
DRY_RUN = False
OVERWRITE_OUTPUT = True

# ── Print active configuration ───────────────────────────────────────────────
print(f'NAIP_ROOTS ({len(NAIP_ROOTS)}):')
for root in NAIP_ROOTS:
    print(f'  {root.resolve()}')
print(f'COG_BLOCK_SIZE:      {COG_BLOCK_SIZE}')
print(f'OVERVIEW_RESAMPLING: {OVERVIEW_RESAMPLING}')
print(f'COG_COMPRESS:        {COG_COMPRESS}  (PREDICTOR={COG_PREDICTOR})')
print(f'BUILD_VIZ_VRTS:      {BUILD_VIZ_VRTS}')
print(f'AGGREGATE_DIR:       {AGGREGATE_DIR.resolve()}')
print(f'DRY_RUN:             {DRY_RUN}')
print(f'OVERWRITE_OUTPUT:    {OVERWRITE_OUTPUT}')

NAIP_ROOTS (1):
  /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip
COG_BLOCK_SIZE:      512
OVERVIEW_RESAMPLING: AVERAGE
COG_COMPRESS:        DEFLATE  (PREDICTOR=2)
BUILD_VIZ_VRTS:      True
AGGREGATE_DIR:       /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs_aggregates
DRY_RUN:             False
OVERWRITE_OUTPUT:    True


## Cell 6: Helper Functions (Part 1)

This first helper cell covers image discovery and spatial metadata math:

- `CogPlan` dataclass
- `iter_images`
- CRS helpers (`make_spatial_ref_from_epsg`, `get_dataset_srs`)
- GSD and zoom helpers (`estimate_gsd_meters`, `zoom_for_full_resolution`)
- overview calculator (`calculate_overview_levels`)

In [3]:
@dataclass
class CogPlan:
    # A single source image and its canonical COG output target.
    image_path: Path
    output_cog: Path
    size_bytes: int
    band_count: int
    width: int
    height: int
    gsd_m: float | None
    native_zoom: int | None
    overview_levels: list[int]
    source_is_cog: bool
    phase: str | None  # 'prefire', 'postfire', or None when unknown


def human_size(num_bytes: int) -> str:
    # Convert bytes into an easy-to-read text label.
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024
    return f'{num_bytes} B'


def iter_images(root_dir: Path) -> Iterable[Path]:
    # Recursively yield all TIFF files under a root folder.
    for p in root_dir.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            yield p


def make_spatial_ref_from_epsg(epsg_code: int) -> osr.SpatialReference:
    # Build an OSR spatial reference object from an EPSG integer.
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(epsg_code)
    srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    return srs


def get_dataset_srs(ds: gdal.Dataset):
    # Prefer the CRS stored in the file header.
    proj_wkt = ds.GetProjection()
    if proj_wkt:
        srs = osr.SpatialReference()
        srs.ImportFromWkt(proj_wkt)
        srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
        return srs

    # If no CRS exists in the file, use the fallback EPSG when configured.
    if FALLBACK_EPSG is not None:
        return make_spatial_ref_from_epsg(FALLBACK_EPSG)
    return None


def pixel_to_geo(gt, px: float, py: float) -> tuple[float, float]:
    # Convert pixel coordinates into map coordinates using GDAL geotransform.
    x = gt[0] + px * gt[1] + py * gt[2]
    y = gt[3] + px * gt[4] + py * gt[5]
    return x, y


def estimate_gsd_meters(ds: gdal.Dataset, src_srs: osr.SpatialReference) -> float | None:
    # Estimate pixel size (meters/pixel) by projecting one-pixel offsets to EPSG:3857.
    gt = ds.GetGeoTransform(can_return_null=True)
    if gt is None:
        return None

    p0 = pixel_to_geo(gt, 0.0, 0.0)
    px = pixel_to_geo(gt, 1.0, 0.0)
    py = pixel_to_geo(gt, 0.0, 1.0)

    web_merc = make_spatial_ref_from_epsg(3857)
    tx = osr.CoordinateTransformation(src_srs, web_merc)
    p0m = tx.TransformPoint(*p0)
    pxm = tx.TransformPoint(*px)
    pym = tx.TransformPoint(*py)

    x_res = math.hypot(pxm[0] - p0m[0], pxm[1] - p0m[1])
    y_res = math.hypot(pym[0] - p0m[0], pym[1] - p0m[1])
    return float((x_res + y_res) / 2.0)


def zoom_for_full_resolution(gsd_m_per_px: float) -> int:
    # Convert GSD to nearest full-resolution Web Mercator zoom.
    if gsd_m_per_px <= 0:
        return 0
    z = math.ceil(math.log2(156543.03392804097 / gsd_m_per_px))
    return max(0, min(24, z))


def calculate_overview_levels(width: int, height: int) -> list[int]:
    # Choose powers-of-2 overviews so the smallest overview fits in one tile block.
    max_dim = max(width, height)
    if max_dim <= COG_BLOCK_SIZE:
        return []
    num_levels = math.ceil(math.log2(max_dim / COG_BLOCK_SIZE))
    return [2 ** i for i in range(1, num_levels + 1)]

In [4]:
def check_is_cog(ds: gdal.Dataset) -> bool:
    # GDAL COGs expose IMAGE_STRUCTURE metadata LAYOUT=COG.
    return ds.GetMetadataItem('LAYOUT', 'IMAGE_STRUCTURE') == 'COG'


def infer_fire_phase(image_path: Path) -> str | None:
    # Classify imagery by folder names in the path.
    parts = {p.lower() for p in image_path.parts}
    if 'prefire' in parts:
        return 'prefire'
    if 'postfire' in parts:
        return 'postfire'
    return None


def find_root_for_image(image_path: Path, roots: list[Path]) -> Path | None:
    # Return the NAIP root that contains the image, if any.
    image_abs = image_path.resolve()
    for root in roots:
        root_abs = root.resolve()
        try:
            image_abs.relative_to(root_abs)
            return root_abs
        except ValueError:
            continue
    return None


def build_cog_from_source(src_image: Path, output_cog: Path, overview_levels: list[int]) -> bool:
    # Build one canonical COG from the original source image, preserving all bands.
    # Returns True when a file is written, or False if skipped due to existing output.
    if output_cog.exists() and not OVERWRITE_OUTPUT:
        return False

    output_cog.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_cog.with_suffix('.building.tif')

    try:
        # Step 1: materialize a tiled intermediate GeoTIFF from the source TIFF.
        tmp_ds = gdal.Translate(
            str(tmp_path),
            str(src_image),
            format='GTiff',
            creationOptions=[
                'TILED=YES',
                f'BLOCKXSIZE={COG_BLOCK_SIZE}',
                f'BLOCKYSIZE={COG_BLOCK_SIZE}',
                f'COMPRESS={COG_COMPRESS}',
                f'PREDICTOR={COG_PREDICTOR}',
                'BIGTIFF=IF_SAFER',
            ],
        )
        if tmp_ds is None:
            raise RuntimeError(f'Failed to create intermediate GeoTIFF: {tmp_path}')

        # Step 2: add overviews so the final COG reads quickly at lower zooms.
        if overview_levels:
            tmp_ds.BuildOverviews(OVERVIEW_RESAMPLING, overview_levels)
        tmp_ds.FlushCache()
        tmp_ds = None

        # Step 3: convert the intermediate into a standards-compliant COG.
        cog_ds = gdal.Translate(
            str(output_cog),
            str(tmp_path),
            format='COG',
            creationOptions=[
                f'COMPRESS={COG_COMPRESS}',
                f'PREDICTOR={COG_PREDICTOR}',
                f'BLOCKSIZE={COG_BLOCK_SIZE}',
                'BIGTIFF=IF_SAFER',
            ],
        )
        if cog_ds is None:
            raise RuntimeError(f'Failed to create COG: {output_cog}')
        cog_ds = None

    finally:
        if tmp_path.exists():
            tmp_path.unlink()

    return True

## Cell 7: Helper Functions (Part 2)

This second helper cell focuses on COG-specific logic and path classification:

- `check_is_cog`
- `infer_fire_phase`
- `find_root_for_image`
- `build_cog_from_source`

## Cell 8: Discover Imagery and Build Canonical COG Plan

This cell scans all configured roots and prepares a build plan that keeps **all bands** from each source TIFF.

For each source image it reports:

- image dimensions, band count, and file size
- estimated GSD in meters/pixel
- native full-resolution web zoom level
- overview factors for multiscale performance
- inferred fire phase (`prefire`/`postfire`) from the path

In [5]:
# Validate configured roots first so errors fail fast.
missing = [r for r in NAIP_ROOTS if not r.exists()]
if missing:
    raise FileNotFoundError(f'NAIP root(s) not found: {[str(m) for m in missing]}')

plans: list[CogPlan] = []
total_source_bytes = 0

# Gather all candidate TIFFs from every configured NAIP root.
all_images = sorted(img for root in NAIP_ROOTS for img in iter_images(root))

for image_path in all_images:
    size_bytes = image_path.stat().st_size
    total_source_bytes += size_bytes

    ds = gdal.Open(str(image_path), gdal.GA_ReadOnly)
    if ds is None:
        print(f'[skip] Cannot open: {image_path}')
        continue

    src_srs = get_dataset_srs(ds)
    if src_srs is None:
        print(f'[skip] {image_path.name} has no CRS and FALLBACK_EPSG is disabled.')
        ds = None
        continue

    width, height = ds.RasterXSize, ds.RasterYSize
    band_count = ds.RasterCount
    source_is_cog = check_is_cog(ds)
    phase = infer_fire_phase(image_path)

    try:
        gsd_m = estimate_gsd_meters(ds, src_srs)
    except Exception as exc:
        print(f'[skip] {image_path.name} metadata error: {exc}')
        continue
    finally:
        ds = None

    native_zoom = zoom_for_full_resolution(gsd_m) if gsd_m else None
    overview_levels = calculate_overview_levels(width, height)

    # Canonical output path: one COG per source image, preserving all source bands.
    output_cog = image_path.parent / COGS_DIRNAME / f'{image_path.stem}.tif'

    plans.append(CogPlan(
        image_path=image_path,
        output_cog=output_cog,
        size_bytes=size_bytes,
        band_count=band_count,
        width=width,
        height=height,
        gsd_m=gsd_m,
        native_zoom=native_zoom,
        overview_levels=overview_levels,
        source_is_cog=source_is_cog,
        phase=phase,
    ))

# ---- Discovery summary for quick QA ------------------------------------------------
already_cog = sum(1 for p in plans if p.source_is_cog)
prefire_count = sum(1 for p in plans if p.phase == 'prefire')
postfire_count = sum(1 for p in plans if p.phase == 'postfire')
unknown_phase = sum(1 for p in plans if p.phase is None)

print('=== Discovery Summary ===')
print(f'Roots scanned:      {len(NAIP_ROOTS)}')
print(f'Images discovered:  {len(plans)}')
print(f'Already COG:        {already_cog} / {len(plans)}')
print(f'Prefire:            {prefire_count}')
print(f'Postfire:           {postfire_count}')
print(f'Unknown phase:      {unknown_phase}')
print(f'Total source size:  {human_size(total_source_bytes)}')
print()

for p in plans:
    gsd_text = f'{p.gsd_m:.4f} m/px' if p.gsd_m else 'unavailable'
    zoom_text = f'z={p.native_zoom}' if p.native_zoom is not None else 'unavailable'
    phase_text = p.phase if p.phase is not None else 'unknown'
    cog_flag = ' [already COG]' if p.source_is_cog else ''

    if p.native_zoom is not None and p.overview_levels:
        ov_labels = [
            f'{factor}x (~z={max(0, p.native_zoom - i)})'
            for i, factor in enumerate(p.overview_levels, start=1)
        ]
    else:
        ov_labels = [f'{f}x' for f in p.overview_levels]

    print(f'Image:        {p.image_path}{cog_flag}')
    print(f'Bands:        {p.band_count}  |  Size: {human_size(p.size_bytes)}')
    print(f'Dimensions:   {p.width} x {p.height} px')
    print(f'Phase:        {phase_text}')
    print(f'GSD / Zoom:   {gsd_text}  |  {zoom_text}')
    print(f'Output COG:   {p.output_cog}')
    print(f'Overviews:    {", ".join(ov_labels) if ov_labels else "none"}')
    print('---')

=== Discovery Summary ===
Roots scanned:      1
Images discovered:  60
Already COG:        0 / 60
Prefire:            29
Postfire:           31
Unknown phase:      0
Total source size:  28.27 GB

Image:        downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/m_3611835_se_11_060_20220707.tif
Bands:        4  |  Size: 482.27 MB
Dimensions:   10210 x 12380 px
Phase:        postfire
GSD / Zoom:   0.7466 m/px  |  z=18
Output COG:   downloads/castle/naip/postfire/m_3611835_se_11_060_20220707/cogs/m_3611835_se_11_060_20220707.tif
Overviews:    2x (~z=17), 4x (~z=16), 8x (~z=15), 16x (~z=14), 32x (~z=13)
---
Image:        downloads/castle/naip/postfire/m_3611835_sw_11_060_20220703/m_3611835_sw_11_060_20220703.tif
Bands:        4  |  Size: 483.13 MB
Dimensions:   10220 x 12390 px
Phase:        postfire
GSD / Zoom:   0.7466 m/px  |  z=18
Output COG:   downloads/castle/naip/postfire/m_3611835_sw_11_060_20220703/cogs/m_3611835_sw_11_060_20220703.tif
Overviews:    2x (~z=17), 4x (~z=16),

Warning 1: m_3611835_se_11_060_20220707.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611835_sw_11_060_20220703.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611837_se_11_060_20220704.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611843_ne_11_060_20220707.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3611843_nw_11_060_20220703.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as

## Cell 10: Build Canonical COGs (Keep All Source Bands)

For each source image in the plan this cell writes exactly one COG that preserves all original source bands.

- Source TIFF with `N` bands -> output COG with the same `N` bands.
- Overview levels are built from image dimensions.
- No RGB/IRG band-dropping happens in this step.

When `DRY_RUN=True`, paths and settings are printed but nothing is written.

In [6]:
cogs_built: list[Path] = []
images_built = 0
images_errored = 0

for p in plans:
    if DRY_RUN:
        ov_str = str(p.overview_levels) if p.overview_levels else '[]'
        print(f'[dry-run] {p.image_path.name}')
        print(f'  source bands: {p.band_count}')
        print(f'  output COG:   {p.output_cog}')
        print(f'  overviews:    {ov_str}')
        cogs_built.append(p.output_cog)
        continue

    try:
        written = build_cog_from_source(p.image_path, p.output_cog, p.overview_levels)
        cogs_built.append(p.output_cog)
        images_built += 1
        status = 'written' if written else 'skipped (exists)'
        print(f'[done] {p.image_path.name} -> {p.output_cog.name} ({status})')
    except Exception as exc:
        images_errored += 1
        print(f'[error] {p.image_path.name}: {exc}')

print()
print('=== Canonical COG Build Summary ===')
if DRY_RUN:
    print(f'Mode: DRY-RUN  |  Images planned: {len(plans)}')
    print('Set DRY_RUN=False and re-run Cells 8, 10, 12, and 14 to write output.')
else:
    print(f'Mode: WRITE  |  Images built: {images_built}  |  Errors: {images_errored}')

Warning 1: m_3611835_se_11_060_20220707.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3611835_se_11_060_20220707.tif -> m_3611835_se_11_060_20220707.tif (written)


Warning 1: m_3611835_sw_11_060_20220703.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3611835_sw_11_060_20220703.tif -> m_3611835_sw_11_060_20220703.tif (written)


Warning 1: m_3611837_se_11_060_20220704.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


...

[done] m_3611853_ne_11_060_20200726.tif -> m_3611853_ne_11_060_20200726.tif (written)


Warning 1: m_3611853_nw_11_060_20200802.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3611853_nw_11_060_20200802.tif -> m_3611853_nw_11_060_20200802.tif (written)


Warning 1: m_3611854_se_11_060_20200726.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3611854_se_11_060_20200726.tif -> m_3611854_se_11_060_20200726.tif (written)

=== Canonical COG Build Summary ===
Mode: WRITE  |  Images built: 60  |  Errors: 0


## Cell 12: Optional Visualization VRTs from Canonical COGs

If `BUILD_VIZ_VRTS=True`, this cell builds two aggregate VRT mosaics from canonical COGs:

- `rgb_all.vrt` using bands `1,2,3`
- `irg_all.vrt` using bands `4,1,2`

This keeps storage efficient (single canonical COGs) while still supporting common visualization styles.

In [7]:
def build_visualization_vrt(cog_list: list[Path], output_vrt: Path, bands: tuple[int, int, int], label: str) -> None:
    # Build one aggregate VRT with a specific visualization band recipe.
    if not cog_list:
        print(f'[skip] No source COGs available for {label}.')
        return

    output_vrt.parent.mkdir(parents=True, exist_ok=True)
    ds = gdal.BuildVRT(
        str(output_vrt),
        [str(p) for p in cog_list],
        options=gdal.BuildVRTOptions(
            resolution='highest',
            allowProjectionDifference=True,
            bandList=list(bands),
        ),
    )
    if ds is None:
        raise RuntimeError(f'gdal.BuildVRT failed for: {output_vrt}')
    ds.FlushCache()
    ds = None
    print(f'[done] {label}: {output_vrt}  ({len(cog_list)} source COGs)')


if not BUILD_VIZ_VRTS:
    print('BUILD_VIZ_VRTS=False -> skipping aggregate RGB/IRG VRT creation.')
elif DRY_RUN:
    print('[dry-run] Aggregate VRT targets:')
    print(f'  RGB: {RGB_AGGREGATE_VRT} using bands {RGB_BANDS}')
    print(f'  IRG: {IRG_AGGREGATE_VRT} using bands {IRG_BANDS}')
    print('Set DRY_RUN=False to write VRT files.')
else:
    existing_cogs = [p for p in cogs_built if p.exists()]
    print(f'Canonical COGs found on disk: {len(existing_cogs)} / {len(cogs_built)}')

    # Only build IRG if there are enough bands in every source image.
    min_bands = min((p.band_count for p in plans), default=0)
    build_visualization_vrt(existing_cogs, RGB_AGGREGATE_VRT, RGB_BANDS, 'RGB mosaic')
    if min_bands >= 4:
        build_visualization_vrt(existing_cogs, IRG_AGGREGATE_VRT, IRG_BANDS, 'IRG mosaic')
    else:
        print('[skip] IRG VRT requires band 4; at least one source image has < 4 bands.')

Canonical COGs found on disk: 60 / 60
[done] RGB mosaic: downloads/castle/naip/cogs_aggregates/rgb_all.vrt  (60 source COGs)
[done] IRG mosaic: downloads/castle/naip/cogs_aggregates/irg_all.vrt  (60 source COGs)


## Recommended Run Order

1. Run the environment preflight cell.
2. Run the configuration cell and confirm printed settings.
3. Run Helper Functions Part 1 and Part 2.
4. Run the discovery/planning cell.
5. Run the canonical COG build cell.
6. Run the optional visualization VRT cell.
7. Run the consolidation/move cell (`cogs/prefire` and `cogs/postfire`).

In [8]:
import shutil

if DRY_RUN:
    print('[dry-run] Consolidation targets (no files moved):')
else:
    print('Moving canonical COGs into root-level prefire/postfire folders...')

moved = 0
skipped_missing = 0
skipped_unknown_phase = 0
skipped_no_root = 0
renamed_collisions = 0

for p in plans:
    phase = p.phase
    if phase is None:
        print(f'[skip] Cannot infer prefire/postfire from source path: {p.image_path}')
        skipped_unknown_phase += 1
        continue

    root = find_root_for_image(p.image_path, NAIP_ROOTS)
    if root is None:
        print(f'[skip] Source image is not under configured roots: {p.image_path}')
        skipped_no_root += 1
        continue

    # Required destination structure under each original root.
    target_dir = root / 'cogs' / phase
    target_dir.mkdir(parents=True, exist_ok=True)

    source_cog = p.output_cog.resolve()
    target_path = target_dir / p.output_cog.name

    if not source_cog.exists():
        print(f'[skip] Missing COG on disk: {p.output_cog}')
        skipped_missing += 1
        continue

    # Avoid accidental overwrite by adding numeric suffixes.
    if target_path.exists() and target_path.resolve() != source_cog:
        stem = target_path.stem
        suffix = target_path.suffix
        n = 1
        while True:
            candidate = target_dir / f'{stem}_{n}{suffix}'
            if not candidate.exists():
                target_path = candidate
                renamed_collisions += 1
                break
            n += 1

    if DRY_RUN:
        print(f'[dry-run] {p.output_cog} -> {target_path}')
        continue

    shutil.move(str(source_cog), str(target_path))
    print(f'[moved] {p.output_cog.name} -> {target_path}')
    moved += 1

print()
print('=== Consolidation Summary ===')
mode = 'DRY-RUN' if DRY_RUN else 'WRITE'
print(f'Mode:                  {mode}')
print(f'Total planned COGs:    {len(plans)}')
print(f'Moved:                 {moved}')
print(f'Skipped (missing):     {skipped_missing}')
print(f'Skipped (no phase):    {skipped_unknown_phase}')
print(f'Skipped (no root):     {skipped_no_root}')
print(f'Renamed collisions:    {renamed_collisions}')

Moving canonical COGs into root-level prefire/postfire folders...
[moved] m_3611835_se_11_060_20220707.tif -> /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs/postfire/m_3611835_se_11_060_20220707.tif
[moved] m_3611835_sw_11_060_20220703.tif -> /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs/postfire/m_3611835_sw_11_060_20220703.tif
[moved] m_3611837_se_11_060_20220704.tif -> /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs/postfire/m_3611837_se_11_060_20220704.tif
[moved] m_3611837_sw_11_060_20220704.tif -> /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs/postfire/m_3611837_sw_11_060_20220704.tif
[moved] m_3611842_ne_11_060_20220704.tif -> /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs/postfire/m_3611842_ne_11_060_20220704.tif
[moved] m_3611842_nw_11_060_20220704.tif -> /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/castle/naip/cogs/post

## Cell 13 Notes: Consolidation Behavior

The move step is intentionally conservative for safety:

- Files are moved only when they exist on disk.
- Files without clear `prefire` or `postfire` in the source path are skipped.
- Name collisions are resolved by appending `_1`, `_2`, ... to the destination filename.
- In dry-run mode, all destination paths are previewed with no filesystem changes.